# 2. Flagging, step by step

Radio-frequency interference (RFI) is man-made emission: satellites, aircraft
transponders, mobile phones. It is usually much brighter than the sky, narrow in
frequency or short in time, and it does not follow the measurement equation. A
**flag** marks a cell (row, channel, correlation) as not to be used. Flags the
data arrive with are treated as an immutable lower bound in this project: no
method may clear them.

MeerKAT pipelines (CARACAL) flag with **Tricolour**, which runs a *strategy*: an
ordered list of steps. Rang ports Tricolour to Rust (research/core-d-tricolour.md)
and can return the flags after every step, so we can see what each step does.

In [ ]:
%matplotlib inline
import os

os.environ.setdefault("RAYON_NUM_THREADS", "4")  # share the cores; set before the first core call

import numpy as np
import pandas as pd

from rangtoy.notebook import data, experiment, flags, gains, image, steer, survey

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)

# Downloads and the extracted fixture go here; choose your own with RANG_NOTEBOOK_CACHE.
CACHE = data.default_cache_dir()
# LUNO sections run when a local copy of the slice exists (RANG_NOTEBOOK_LUNO=0 turns them off).
LUNO = os.environ.get("RANG_NOTEBOOK_LUNO", "1") != "0" and data.find_luno_slice() is not None
print("cache:", CACHE, "| LUNO slice available:", LUNO)

In [ ]:
ds = data.load_fixture(cache_dir=CACHE)
pd.Series(data.summary(ds), name="value").to_frame()

## Strategies

The teaching strategy is short and built from Tricolour's task list. CARACAL's
shipped strategies are read from their YAML files when a copy is available
(`RANG_STRATEGY_DIR`, or the project's `outputs/caracal-strategy-inputs-8f7bb363`).

In [ ]:
library = flags.strategy_library()
print("strategies:", list(library))
flags.describe_strategy(library["teaching"])

**SumThreshold** (Offringa et al. 2010) is the workhorse. It smooths the data to
estimate a background, then looks for runs of 1, 2, 4, 8 … consecutive cells
whose *summed* excess over the background is above a threshold that falls as
the run gets longer: a single very bright cell is flagged, and so is a long run
of moderately bright ones. It works on the amplitude of the four correlations
combined, per baseline, in time and in frequency.

In [ ]:
run = flags.run_strategy(ds, library["teaching"])
print(f"core time {run['seconds']:.2f} s")
flags.step_table(run)

`step_table` counts window cells (baseline × integration × channel, cross
baselines only): the fraction flagged after each step, how many cells the step
newly flagged and how many it cleared. The drawing below shows one step on one
baseline: before, after, and the difference (red: newly flagged; blue: cleared).
**What to look for:** does SumThreshold pick out isolated features or whole
bands? Does it flag where the waterfall in notebook 1 showed stripes?

In [ ]:
flags.step_waterfall(ds, run, "02-sum_threshold", ds.baselines()[0])

## A CARACAL strategy

The calibrator first-pass strategy is what the campaigns compare against. It has
twelve steps, including a `uvcontsub_flagger` that fits and subtracts a smooth
spectrum, and steps that *replace* the accumulated flags rather than add to them;
the final `combine_with_input_flags` puts the original flags back. Watch the
"cleared" column: a step that clears flags is not a bug, it is the strategy
discarding an intermediate state.

In [ ]:
name = "caracal-calibrator-firstpass"
if name in library:
    display(flags.describe_strategy(library[name]))
    caracal = flags.run_strategy(ds, library[name])
    display(flags.step_table(caracal))
    display(flags.step_waterfall(ds, caracal, 3, ds.baselines()[1]))
else:
    caracal = None
    print("skipped: CARACAL strategy files not found (set RANG_STRATEGY_DIR)")

## What the strategy would remove from training

In the campaigns, a flagging method is judged by what happens to calibration
and imaging *when its flags are added to the training cells*. These are the
"additions":

In [ ]:
final = caracal if caracal is not None else run
display(flags.additions_by_antenna(ds, final))
flags.flag_fraction_by_channel(ds, final["flags"])

### What a professional checks

- **Superset**: the strategy's flags must include every original flag.
- **Uniform over correlations**: Tricolour flags all four correlations together.
- **Where the new flags are**: a strategy that removes a third of the clean
  training cells on a calibrator is not "finding RFI"; on four stations
  SumThreshold's statistics are weak, and its settings were tuned for 64 dishes.

In [ ]:
original = ds.state["original_flags"]
pd.Series({
    "superset of the original flags": not np.any(original & ~final["flags"]),
    "uniform over correlations": bool(np.all(final["flags"] == final["flags"][..., :1])),
    "train cells added": int(final["additions_on_train"].sum()),
    "share of train": float(final["additions_on_train"].sum() / ds.state["train"].sum()),
})

## Steer it yourself

Pick a strategy, a step and a baseline. Each strategy runs once and is cached;
moving the step slider only redraws. (In a static copy of this notebook the
widget is inactive; the cell after it shows the same panel's output for one
setting.)

In [ ]:
panel = steer.FlagPanel(ds, library)
panel

In [ ]:
result = panel.update("teaching", step=2, baseline=ds.baselines()[2])
result["waterfall"]